<a href="https://colab.research.google.com/github/roshankumal23/Ideating-Porotype/blob/master/model-stage2/Misinformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

DATA_DIR = '/content/drive/MyDrive/Misinformation'

if os.path.exists(DATA_DIR):
    print(f"Files in '{DATA_DIR}':")
    for item in os.listdir(DATA_DIR):
        print(item)
else:
    print(f"The directory '{DATA_DIR}' does not exist. Please ensure your Google Drive is mounted and the path is correct.")

In [ ]:
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/Misinformation'

df1 = pd.read_csv(f'{DATA_DIR}/ai_misinfo_v2 (1).csv')
df2 = pd.read_csv(f'{DATA_DIR}/hybrid_ai_misinfo_50k (1).csv')

print(f"Dataset 1: {df1.shape}")
print(f"Dataset 2: {df2.shape}")

df = pd.concat([df1, df2], ignore_index=True)
print(f"Merged   : {df.shape}")
print(f"Columns  : {df.columns.tolist()}")

In [ ]:

TEXT_COL  = 'text'
LABEL_COL = 'final_label'

print(f"Before cleaning : {df.shape}")
print(f"Duplicates      : {df.duplicated().sum()}")
print(f"Null values     :\n{df[[TEXT_COL, LABEL_COL]].isnull().sum()}")

# Remove exact duplicate rows
df.drop_duplicates(inplace=True)

# Remove rows with missing text or label
df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)

# Remove rows where text is empty or whitespace only
df = df[df[TEXT_COL].str.strip() != '']

df.reset_index(drop=True, inplace=True)
print(f"\nAfter cleaning  : {df.shape}")
print(f"Label distribution:\n{df[LABEL_COL].value_counts()}")

In [ ]:
import re

def preprocess_text(text):
    if not isinstance(text, str):
        return ''

    # Normalize URLs to token (as per your preprocessing pipeline)
    text = re.sub(r'http\S+|www\.\S+', '[URL]', text)

    # Normalize emails to token
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove special characters, keep letters, numbers, spaces
    text = re.sub(r'[^a-zA-Z0-9\s\[\]]', '', text)

    # Normalize multiple spaces to single space
    text = re.sub(r'\s+', ' ', text).strip()

    # Lowercase
    text = text.lower()

    return text

df['clean_text'] = df[TEXT_COL].apply(preprocess_text)

# Drop any rows that became empty after cleaning
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)

print(f"Shape after preprocessing: {df.shape}")
print("\nSample:")
print(df[[TEXT_COL, 'clean_text']].head(3).to_string())

In [ ]:
# Save cleaned file in the same Google Drive folder
output_path = f"{DATA_DIR}/merged_cleaned_misinfo.csv"

df.to_csv(output_path, index=False, encoding="utf-8")

print("File saved successfully at:")
print(output_path)